### Limpieza SMQ_H


In [1]:
import pandas as pd
import numpy as np
# Cargar archivo con columnas SMQ seleccionadas
smq = pd.read_csv("../data/interim/smq_h_columnas.csv")
df_demo = pd.read_csv("../data/interim/demo_h_columnas.csv")
df_edad = df_demo[['SEQN', 'RIDAGEYR']].copy()
smq = smq.merge(df_edad,on='SEQN',how='inner')
smq = smq[smq['RIDAGEYR']>=18]
# Vista inicial
smq.info()


<class 'pandas.core.frame.DataFrame'>
Index: 6113 entries, 0 to 7167
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   SEQN      6113 non-null   float64
 1   SMQ040    2579 non-null   float64
 2   SMD641    1232 non-null   float64
 3   SMD650    1219 non-null   float64
 4   SMQ078    970 non-null    float64
 5   SMQ020    6113 non-null   float64
 6   SMQ621    0 non-null      float64
 7   SMD630    0 non-null      float64
 8   SMD030    2579 non-null   float64
 9   SMQ670    1232 non-null   float64
 10  SMQ848    583 non-null    float64
 11  SMQ852Q   580 non-null    float64
 12  SMQ852U   577 non-null    float64
 13  RIDAGEYR  6113 non-null   float64
dtypes: float64(14)
memory usage: 716.4 KB


In [2]:
smq.isna().sum()

SEQN           0
SMQ040      3534
SMD641      4881
SMD650      4894
SMQ078      5143
SMQ020         0
SMQ621      6113
SMD630      6113
SMD030      3534
SMQ670      4881
SMQ848      5530
SMQ852Q     5533
SMQ852U     5536
RIDAGEYR       0
dtype: int64

In [3]:
SMQ_COLS = [
    "SEQN",
    "SMQ040",
    "SMD641",
    "SMD650",
    "SMQ078",
    "SMQ020",
    # "SMQ621", solo esta dirigda a menores de edad
    # "SMD630", solo esta dirigda a menores de edad
    "SMD030",
    "SMQ670",
    "SMQ848",
    "SMQ852Q",
    "SMQ852U"
]

smq = smq[SMQ_COLS].copy()
smq.head()


,SEQN,SMQ040,SMD641,SMD650,SMQ078,SMQ020,SMD030,SMQ670,SMQ848,SMQ852Q,SMQ852U
0,73557.0,3.0,NaN,NaN,NaN,1.0,1.700000e+01,NaN,NaN,NaN,NaN
1,73558.0,2.0,1.0,1.0,NaN,1.0,5.397605e-79,2.0,NaN,NaN,NaN
2,73559.0,3.0,NaN,NaN,NaN,1.0,2.000000e+01,NaN,NaN,NaN,NaN
3,73561.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN
4,73562.0,3.0,NaN,NaN,NaN,1.0,1.800000e+01,NaN,NaN,NaN,NaN


In [4]:
codigos_invalidos = [7, 9, 77, 99, 777, 999, 7777, 9999]
total_a_reemplazar = smq.isin(codigos_invalidos).sum().sum()
print(f"Se van a reemplazar {total_a_reemplazar} valores en total.")

smq.replace(codigos_invalidos, np.nan, inplace=True)

smq.isna().sum()


Se van a reemplazar 176 valores en total.


SEQN          0
SMQ040     3534
SMD641     4890
SMD650     4944
SMQ078     5175
SMQ020        2
SMD030     3579
SMQ670     4881
SMQ848     5537
SMQ852Q    5564
SMQ852U    5536
dtype: int64

In [5]:
for col in smq.columns:
    if col != "SEQN":
        smq[col] = pd.to_numeric(smq[col], errors="coerce")

smq.dtypes
smq.isna().sum()



SEQN          0
SMQ040     3534
SMD641     4890
SMD650     4944
SMQ078     5175
SMQ020        2
SMD030     3579
SMQ670     4881
SMQ848     5537
SMQ852Q    5564
SMQ852U    5536
dtype: int64

In [6]:
cols_binarias = ["SMQ020", "SMQ670"]

for col in cols_binarias:
    smq[col] = smq[col].map({1: 1, 2: 0})

smq[cols_binarias].head()


,SMQ020,SMQ670
0,1.0,NaN
1,1.0,0.0
2,1.0,NaN
3,0.0,NaN
4,1.0,NaN


In [7]:
smq.loc[(smq["SMD641"] < 0) | (smq["SMD641"] > 30), "SMD641"] = np.nan
smq.loc[(smq["SMD650"] < 1) | (smq["SMD650"] > 95), "SMD650"] = np.nan
smq.loc[(smq["SMD030"] < 0) | (smq["SMD030"] > 80), "SMD030"] = np.nan
smq.loc[(smq["SMQ848"] < 1) | (smq["SMQ848"] > 20), "SMQ848"] = np.nan
smq.loc[(smq["SMQ852Q"] < 0) | (smq["SMQ852Q"] > 356), "SMQ852Q"] = np.nan

smq.describe()


,SEQN,SMQ040,SMD641,SMD650,SMQ078,SMQ020,SMD030,SMQ670,SMQ848,SMQ852Q,SMQ852U
count,6113.000000,2579.000000,1.223000e+03,1169.000000,938.000000,6111.000000,2.534000e+03,1232.000000,576.000000,5.490000e+02,577.000000
mean,78684.741862,2.137650,2.583974e+01,10.697177,2.375267,0.422026,1.725770e+01,0.473214,3.302083,4.493625e+00,1.786828
std,2920.806912,0.942517,8.441218e+00,8.588320,1.224050,0.493923,6.376920e+00,0.499485,3.641175,1.614280e+01,0.846541
min,73557.000000,1.000000,5.397605e-79,1.000000,1.000000,0.000000,5.397605e-79,0.000000,1.000000,5.397605e-79,1.000000
25%,76180.000000,1.000000,3.000000e+01,4.000000,1.000000,0.000000,1.500000e+01,0.000000,1.000000,1.000000e+00,1.000000
50%,78735.000000,3.000000,3.000000e+01,10.000000,2.000000,0.000000,1.700000e+01,0.000000,2.000000,2.000000e+00,2.000000
75%,81179.000000,3.000000,3.000000e+01,15.000000,3.000000,1.000000,2.000000e+01,1.000000,4.000000,4.000000e+00,3.000000
max,83729.000000,3.000000,3.000000e+01,90.000000,6.000000,1.000000,6.400000e+01,1.000000,20.000000,3.560000e+02,3.000000


In [8]:
# si no fumas 100 cigarillos en toda tu vida te saltas la entrevista

#  marcar 3 (not all all) en SMQ040 - Do you now smoke cigarettes
smq.loc[smq["SMQ020"] == 0, "SMQ040"] = 3
# 0 en SMD650 - Avg # cigarettes/day during past 30 days
smq.loc[smq["SMQ020"] == 0, "SMD650"] = 0
# 0 en SMD641 - # days smoked cigs during past 30 days
smq.loc[smq["SMQ020"] == 0, "SMD641"] = 0
# 0 en SMQ078 - How soon after waking do you smoke
smq.loc[smq["SMQ020"] == 0, "SMQ078"] = 0
# 0 (never smoke regularly)  SMD030 - Age started smoking cigarettes regularly
smq.loc[smq["SMQ020"] == 0, "SMD030"] = 0
# 0 (No) en SMQ670 - Tried to quit smoking  (convertimos 2 (no) a 0 arriba)
smq.loc[smq["SMQ020"] == 0, "SMQ670"] = 0
# 0 SMQ848 - # times stopped smoking cigarettes
smq.loc[smq["SMQ020"] == 0, "SMQ848"] = 0
# SMQ852Q - How long were you able to stop smoking
smq.loc[smq["SMQ020"] == 0, "SMQ852Q"] = 0
# SMQ852U - Unit of measure (day/week/month/year)
smq.loc[smq["SMQ020"] == 0, "SMQ852U"] = 0


# si respondes 2,3 en SMQ040, no te preguntan SMQ078
# A las personas que respondieron 2 ("Some days") o 3 ("Not at all") en SMQ040, 
# les asignamos un nuevo valor en SMQ078  How soon after waking do you smoke , por ejemplo 0.
condicion = smq['SMQ040'].isin([2, 3])
smq.loc[condicion & smq['SMQ078'].isna(), 'SMQ078'] = 0

In [9]:

# # 6
# # 6.1 No fuma actualmente → consumo = 0
smq.loc[smq["SMQ040"] == 3, "SMD641"] = 0
smq.loc[smq["SMQ040"] == 3, "SMD650"] = 0
smq.loc[smq["SMQ040"] == 3, "SMQ670"] = 0
smq.loc[smq["SMQ040"] == 3, "SMQ848"] = 0
smq.loc[smq["SMQ040"] == 3, "SMD641"] = 0
smq.loc[smq["SMQ040"] == 3, "SMQ852Q"] = 0
smq.loc[smq["SMQ040"] == 3, "SMQ852U"] = 0


# # 6.4 Intentos de dejar
smq.loc[smq["SMQ670"] == 0, ["SMQ848", "SMQ852Q", "SMQ852U"]] = 0



In [10]:
smq.isna().sum()


SEQN        0
SMQ040      2
SMD641     11
SMD650     65
SMQ078     56
SMQ020      2
SMD030     47
SMQ670      2
SMQ848      9
SMQ852Q    39
SMQ852U    11
dtype: int64

In [11]:
# -------------------------------------------------
# 7. Unir SMQ852Q (cantidad) y SMQ852U (unidad)
# Convertimos todo a días
# -------------------------------------------------

def convertir_a_dias(row):
    cantidad = row["SMQ852Q"]
    unidad = row["SMQ852U"]
    
    if pd.isna(cantidad) or pd.isna(unidad):
        return np.nan
    
    if unidad == 0:
        return 0
    elif unidad == 1:   # días
        return cantidad
    elif unidad == 2:   # semanas
        return cantidad * 7
    elif unidad == 3:   # meses
        return cantidad * 30
    elif unidad == 4:   # años
        return cantidad * 365
    else:
        return np.nan

smq["SMQ852_dias"] = smq.apply(convertir_a_dias, axis=1)

# Eliminamos las columnas originales
smq.drop(columns=["SMQ852Q", "SMQ852U"], inplace=True)

In [12]:
smq = smq.dropna()

In [13]:
import os

# Crear carpeta si no existe
os.makedirs("../data/processed", exist_ok=True)

# Guardar archivo limpio
smq.to_csv("../data/processed/smq_limpio.csv", index=False)

print("SMQ limpio guardado en data/processed/smq_limpio.csv")


SMQ limpio guardado en data/processed/smq_limpio.csv


### Limpieza OCQ_H

In [6]:
# Cargar datos OCQ
import pandas as pd
import numpy as np
ocq = pd.read_csv("../data/interim/ocq_h_columnas.csv")
df_demo = pd.read_csv("../data/interim/demo_h_columnas.csv")
df_edad = df_demo[['SEQN', 'RIDAGEYR']].copy()
ocq = ocq.merge(df_edad,on='SEQN',how='inner')
ocq = ocq[ocq['RIDAGEYR']>=18]

ocq.head()


,SEQN,OCD150,OCQ180,OCQ210,OCD231,OCD241,OCQ260,OCD270,OCD390G,OCD391,OCD392,OCD395,RIDAGEYR
0,73557.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.0,16.0,204.0,69.0
1,73558.0,1.0,50.0,NaN,8.0,16.0,1.0,420.0,2.0,NaN,NaN,NaN,54.0
2,73559.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,21.0,2.0,216.0,72.0
3,73561.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,15.0,8.0,372.0,73.0
4,73562.0,1.0,56.0,NaN,9.0,17.0,2.0,372.0,2.0,NaN,NaN,NaN,56.0


In [7]:
OCQ_COLS = [
    "SEQN",
    'OCD150',
    "OCQ180",   # Horas trabajadas la semana pasada
    "OCQ210",   # Trabaja ≥35h (1=Sí, 2=No)
    "OCD231",   # Industria actual
    "OCD241",   # Ocupación actual
    "OCQ260",   # Tipo de empleo
    "OCD270",   # meses en el trabajo actual
    "OCD390G",  # Tipo de trabajo más largo
    "OCD391",   # Industria trabajo más largo
    "OCD392",   # Ocupación trabajo más largo
    "OCD395"    # Meses en el trabajo más largo
]

ocq = ocq[OCQ_COLS].copy()
ocq.head()

,SEQN,OCD150,OCQ180,OCQ210,OCD231,OCD241,OCQ260,OCD270,OCD390G,OCD391,OCD392,OCD395
0,73557.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,10.0,16.0,204.0
1,73558.0,1.0,50.0,NaN,8.0,16.0,1.0,420.0,2.0,NaN,NaN,NaN
2,73559.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,21.0,2.0,216.0
3,73561.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,15.0,8.0,372.0
4,73562.0,1.0,56.0,NaN,9.0,17.0,2.0,372.0,2.0,NaN,NaN,NaN


In [ ]:
# no remplazar todo
# codigos_invalidos = [7, 9, 77, 99, 777, 999, 77777, 99999]
# ocq.replace(codigos_invalidos, np.nan, inplace=True)

# ocq.isna().sum()


In [8]:
not_at_work = ocq['OCD150'] == 2
# saltas esta pregunta
ocq.loc[not_at_work, 'OCQ180'] = 0

looking_for_work_or_not_working = ocq['OCD150'].isin([3, 4])
# saltar preguntas
ocq.loc[looking_for_work_or_not_working, 'OCQ180'] = 0
ocq.loc[looking_for_work_or_not_working, 'OCQ210'] = 2 # no
ocq.loc[looking_for_work_or_not_working, 'OCD231'] = 0
ocq.loc[looking_for_work_or_not_working, 'OCD241'] = 0
ocq.loc[looking_for_work_or_not_working, 'OCQ260'] = 0


In [9]:

# 1. Definimos la condición (máscara)
same_as_current = ocq['OCD390G'] == 2
# Mapeo: OCD231 -> OCD391 (Industria)
ocq.loc[same_as_current, 'OCD391'] = ocq.loc[same_as_current, 'OCD231']
# Mapeo: OCD241 -> OCD392 (Ocupación)
ocq.loc[same_as_current, 'OCD392'] = ocq.loc[same_as_current, 'OCD241']
# Mapeo: OCD270 -> OCD395 (Meses en el trabajo)
ocq.loc[same_as_current, 'OCD395'] = ocq.loc[same_as_current, 'OCD270']

In [10]:
# Definimos las condiciones 
never_worked = ocq['OCD390G'] == 4
invalid = ocq['OCD390G'].isin([9, 7])
# Usamos '|' para el "OR" lógico entre Series
skip = never_worked | invalid
# Aplicamos los cambios con .loc
ocq.loc[skip, 'OCD391'] = 0
ocq.loc[skip, 'OCD392'] = 0
ocq.loc[skip, 'OCD395'] = 0

In [11]:
# limpieza valores invalidos
invalid_codes = [77777, 99999]
ocq = ocq[~ocq['OCQ180'].isin(invalid_codes)].copy()

invalid_ocq210 = [7, 9]
ocq = ocq[~ocq['OCQ210'].isin(invalid_ocq210)].copy()

invalid_ocq260 = [77, 99]
ocq = ocq[~ocq['OCQ260'].isin(invalid_ocq260)].copy()

invalid_ocd390g = [7, 9]
ocq = ocq[~ocq['OCD390G'].isin(invalid_ocd390g)].copy()

invalid_ocd395 = [77777, 99999]
ocq = ocq[~ocq['OCD395'].isin(invalid_ocd395)].copy()



In [12]:
# ocd270 no estaba evaluada, dropear columna
ocq.drop(columns=['OCD270'], inplace=True)

In [13]:

# si trabajas mas de 35 horas, no aplicas (respondes si)
filtro_mas_35h = ocq['OCQ180'] >= 35
ocq.loc[filtro_mas_35h, 'OCQ210'] = 1 #si

In [14]:
ocq["OCQ210"] = ocq["OCQ210"].map({1: 1, 2: 0})

ocq["OCQ210"].value_counts(dropna=False)


OCQ210
0.0    3360
1.0    2724
NaN       5
Name: count, dtype: int64

In [15]:
# ocq.loc[(ocq["OCQ180"] < 1) | (ocq["OCQ180"] > 120), "OCQ180"] = np.nan
# ocq.loc[(ocq["OCD231"] < 1) | (ocq["OCD231"] > 22), "OCD231"] = np.nan
# ocq.loc[(ocq["OCD241"] < 1) | (ocq["OCD241"] > 23), "OCD241"] = np.nan
# ocq.loc[(ocq["OCQ260"] < 1) | (ocq["OCQ260"] > 6), "OCQ260"] = np.nan
# ocq.loc[(ocq["OCD390G"] < 1) | (ocq["OCD390G"] > 4), "OCD390G"] = np.nan
# ocq.loc[(ocq["OCD391"] < 1) | (ocq["OCD391"] > 22), "OCD391"] = np.nan
# ocq.loc[(ocq["OCD392"] < 1) | (ocq["OCD392"] > 23), "OCD392"] = np.nan
# ocq.loc[(ocq["OCD395"] < 0) | (ocq["OCD395"] > 720), "OCD395"] = np.nan

ocq.describe()


,SEQN,OCD150,OCQ180,OCQ210,OCD231,OCD241,OCQ260,OCD390G,OCD391,OCD392,OCD395
count,6089.000000,6084.000000,6084.000000,6084.000000,6064.000000,6067.000000,6084.000000,6084.000000,6058.000000,6063.000000,6084.000000
mean,78683.377730,2.326101,21.378863,0.447732,6.627144,7.246745,0.836621,1.566897,11.196104,12.887349,144.073471
std,2920.088259,1.458338,22.571856,0.497301,7.120461,8.132877,1.162853,0.762401,5.999904,6.905691,142.280253
min,73557.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,76180.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,6.000000,8.000000,30.000000
50%,78733.000000,1.000000,16.000000,0.000000,5.000000,2.000000,1.000000,1.000000,11.000000,15.000000,96.000000
75%,81178.000000,4.000000,40.000000,1.000000,14.000000,16.000000,1.000000,2.000000,16.000000,19.000000,228.000000
max,83729.000000,4.000000,120.000000,1.000000,22.000000,23.000000,6.000000,4.000000,22.000000,23.000000,720.000000


In [ ]:
#7 --------------------------------------------
# IMPUTACIÓN 
# --------------------------------------------

# # 1 Si nunca trabajó (OCD390G == 4)
# ocq.loc[ocq["OCD390G"] == 4, "OCD395"] = 0
# ocq.loc[ocq["OCD390G"] == 4, "OCQ180"] = 0

# # 2 Variables categóricas → 0 = No aplica
# vars_categoricas = [
#     "OCD231", "OCD241",
#     "OCQ260", "OCD390G", "OCD391", "OCD392"
# ]

# for col in vars_categoricas:
#     ocq[col] = ocq[col].fillna(0)

# # 3 Variable binaria
# ocq["OCQ210"] = ocq["OCQ210"].fillna(0)

# # 4 Continuas → solo lo realmente desconocido
# vars_continuas = ["OCQ180", "OCD395"]

# for col in vars_continuas:
#     ocq[col] = ocq[col].fillna(ocq[col].median())

# ocq.isna().sum()


SEQN       1
OCQ180     0
OCQ210     0
OCD231     0
OCD241     0
OCQ260     0
OCD390G    0
OCD391     0
OCD392     0
OCD395     0
dtype: int64

In [19]:
# revisamos nulos
ocq.isna().sum()

SEQN       0
OCD150     0
OCQ180     0
OCQ210     0
OCD231     0
OCD241     0
OCQ260     0
OCD390G    0
OCD391     0
OCD392     0
OCD395     0
dtype: int64

In [ ]:

# son pocos nulos, dropeamos
ocq.dropna(inplace=True)
ocq.to_csv("../data/processed/ocq_limpio.csv", index=False)

In [18]:
import os

os.makedirs("../data/processed", exist_ok=True)

ocq.to_csv("../data/processed/ocq_limpio.csv", index=False)

print("OCQ limpio guardado correctamente en data/processed/ocq_limpio.csv")


OCQ limpio guardado correctamente en data/processed/ocq_limpio.csv


### Limpieza DLQ_H

In [22]:
# ==========================================
# LIMPIEZA DLQ (Discapacidades y dificultades)
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

# 1) Cargar dataset y seleccionar columnas relevantes
dlq = pd.read_csv("../data/interim/dlq_h_columnas.csv")

DLQ_COLS = [
    "SEQN",
    "DLQ010",  # Dificultad para oír
    "DLQ020",  # Dificultad para ver
    "DLQ040",  # Dificultad para concentrarse/recordar
    "DLQ050",  # Dificultad para caminar/escaleras
    "DLQ060",  # Dificultad para vestirse/bañarse
    "DLQ080"   # Dificultad para hacer mandados solo
]

dlq = dlq[DLQ_COLS].copy()
dlq.head()


,SEQN,DLQ010,DLQ020,DLQ040,DLQ050,DLQ060,DLQ080
0,73557.0,2.0,2.0,2.0,2.0,2.0,2.0
1,73558.0,2.0,2.0,2.0,2.0,2.0,2.0
2,73559.0,1.0,2.0,2.0,2.0,2.0,2.0
3,73560.0,2.0,2.0,2.0,2.0,2.0,NaN
4,73561.0,2.0,2.0,2.0,2.0,2.0,2.0


In [23]:
# 2) Reemplazar códigos inválidos NHANES (Refused / Don't know)
dlq.replace([7, 9], np.nan, inplace=True)


In [24]:
# 3) Convertir variables a numérico
for col in DLQ_COLS:
    if col != "SEQN":
        dlq[col] = pd.to_numeric(dlq[col], errors="coerce")


In [25]:
# 4) Binarizar variables (Sí=1, No=0)
for col in DLQ_COLS:
    if col != "SEQN":
        dlq[col] = dlq[col].map({1: 1, 2: 0})


In [26]:
# 5) Imputación coherente (según lógica de Gabriel)
for col in DLQ_COLS:
    if col != "SEQN":
        dlq[col] = dlq[col].fillna(0)



In [27]:
# 6) Crear carpeta processed si no existe
output_path = Path("../data/processed")
output_path.mkdir(parents=True, exist_ok=True)


In [28]:
# 7) Guardar dataset limpio
dlq.to_csv(output_path / "dlq_limpio.csv", index=False)

print("✅ DLQ limpio guardado en data/processed/dlq_limpio.csv")
dlq.head()


✅ DLQ limpio guardado en data/processed/dlq_limpio.csv


,SEQN,DLQ010,DLQ020,DLQ040,DLQ050,DLQ060,DLQ080
0,73557.0,0.0,0.0,0.0,0.0,0.0,0.0
1,73558.0,0.0,0.0,0.0,0.0,0.0,0.0
2,73559.0,1.0,0.0,0.0,0.0,0.0,0.0
3,73560.0,0.0,0.0,0.0,0.0,0.0,0.0
4,73561.0,0.0,0.0,0.0,0.0,0.0,0.0


### Limpieza SLQ_H

In [29]:
# ==========================================
# LIMPIEZA SLQ (Trastornos del sueño)
# ==========================================

import pandas as pd
import numpy as np
from pathlib import Path

# 1) Cargar dataset y seleccionar columnas
slq = pd.read_csv("../data/interim/slq_h_columnas.csv")

SLQ_COLS = ["SEQN", "SLD010H", "SLQ050", "SLQ060"]
slq = slq[SLQ_COLS].copy()

slq.head()


,SEQN,SLD010H,SLQ050,SLQ060
0,73557.0,7.0,1.0,2.0
1,73558.0,9.0,2.0,2.0
2,73559.0,8.0,2.0,2.0
3,73561.0,9.0,2.0,2.0
4,73562.0,5.0,2.0,1.0


In [30]:
#2 Reemplazar códigos inválidos NHANES
slq["SLD010H"] = slq["SLD010H"].replace([77, 99], np.nan)
slq[["SLQ050", "SLQ060"]] = slq[["SLQ050", "SLQ060"]].replace([7, 9], np.nan)


In [31]:
#3 Convertir variables a formato numerico
for col in SLQ_COLS:
    if col != "SEQN":
        slq[col] = pd.to_numeric(slq[col], errors="coerce")


In [32]:
#4 Limpieza de horas de sueño según codebooK
# Valores válidos: 2–11 horas y 12 = "12 horas o más"
slq["SLD010H"] = slq["SLD010H"].clip(lower=2, upper=12)

# Imputación con mediana
slq["SLD010H"] = slq["SLD010H"].fillna(slq["SLD010H"].median())


In [33]:
#5 Binarizar preguntas SI/NO
slq["SLQ050"] = slq["SLQ050"].map({1: 1, 2: 0})
slq["SLQ060"] = slq["SLQ060"].map({1: 1, 2: 0})


In [34]:
#6 Eliminar pocos casos con NaN restantes
slq = slq.dropna()


In [35]:
slq.to_csv("../data/processed/slq_limpio.csv", index=False)
print("SLQ limpio guardado en data/processed/slq_limpio.csv")


SLQ limpio guardado en data/processed/slq_limpio.csv
